In [3]:
!pip install tinkoff-investments

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.


In [19]:
from tinkoff.invest import (
    Client,
    InstrumentIdType,
    PortfolioResponse,
    MoneyValue,
    Quotation
)
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime, timedelta
import pandas as pd
from google.colab import userdata
TOKEN = userdata.get("TINKOFF_TOKEN")


In [20]:
# === ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ===
def moneyvalue_to_float(mv: MoneyValue) -> float:
    """Преобразует MoneyValue в float"""
    if mv is None:
        return 0.0
    return float(mv.units) + float(mv.nano) / 1e9

def quotation_to_float(q: Quotation) -> float:
    """Преобразует Quotation в float"""
    if q is None:
        return 0.0
    return float(q.units) + float(q.nano) / 1e9

def get_asset_type(instrument_type: str) -> str:
    """Конвертирует тип инструмента Tinkoff в понятный формат"""
    mapping = {
        "share": "Акция",
        "bond": "Облигация",
        "etf": "ETF",
        "currency": "Валюта",
        "future": "Фьючерс",
        "option": "Опцион"
    }
    return mapping.get(instrument_type, "Другое")

In [7]:
def get_dividends(client, figi: str) -> float:
    """Получает дивиденды за последний год"""
    end_date = datetime.utcnow()
    start_date = end_date - timedelta(days=365)

    dividends = client.instruments.get_dividends(
        figi=figi,
        from_=start_date,
        to=end_date
    )

    return sum(
        moneyvalue_to_float(div.dividend_net)
        for div in dividends.dividends
    )

def get_coupons(client, figi: str) -> float:
    """Получает купонный доход за последний год"""
    end_date = datetime.utcnow()
    start_date = end_date - timedelta(days=365)

    coupons = client.instruments.get_bond_coupons(
        figi=figi,
        from_=start_date,
        to=end_date
    )

    return sum(
        moneyvalue_to_float(coupon.pay_one_bond)
        for coupon in coupons.events
    )

In [8]:
 def getPortfolioInfo():
    positions = []
    with Client(TOKEN) as client:
        accounts = client.users.get_accounts()
        portfolio = client.operations.get_portfolio(account_id=accounts.accounts[0].id)
        positions = []

        for position in portfolio.positions:
            # Пропускаем позиции с нулевым количеством
            quantity = quotation_to_float(position.quantity)
            if quantity == 0:
                continue

            # Получаем информацию об инструменте
            try:
                instrument = client.instruments.get_instrument_by(
                    id_type=InstrumentIdType.INSTRUMENT_ID_TYPE_FIGI,
                    id=position.figi
                )
                instr = instrument.instrument
            except Exception as e:
                print(f"Ошибка получения информации об инструменте {position.figi}: {e}")
                continue

            # В новой версии API quantity уже содержит реальное количество (уже с учетом лота)
            # Нет необходимости умножать на instr.lot
            real_quantity = quantity

            # Данные о дивидендах/купоне
            dividends = 0
            if instr.instrument_type == "share":
                dividends = get_dividends(client, position.figi)
            elif instr.instrument_type == "bond":
                dividends = get_coupons(client, position.figi)

            # Текущая цена может быть None в некоторых случаях
            current_price = moneyvalue_to_float(position.current_price) if position.current_price else 0

            positions.append({
                "Тип актива": get_asset_type(instr.instrument_type),
                "Название": instr.name,
                "Тикер": instr.ticker,
                "ISIN": instr.isin,
                "Количество": real_quantity,
                "Средняя цена": moneyvalue_to_float(position.average_position_price),
                "Текущая цена": current_price,
                "Валюта": position.average_position_price.currency if position.average_position_price else "RUB",
                "Годовой дивиденд/купон": dividends,
                "Дивидендная/купонная доходность": dividends / current_price if current_price > 0 else 0,
                "Прогнозируемый годовой доход": dividends * real_quantity,
                "Текущая стоимость": current_price * real_quantity,
                "Прибыль/убыток": (current_price - moneyvalue_to_float(position.average_position_price)) * real_quantity
            })
        return positions

In [9]:
positions = getPortfolioInfo()
#positions

In [11]:
SPREADSHEET_NAME="1lcRkRMJg-4sdvvHUj0phIBb4BzlIpJUW"

In [22]:
def update_portfolio_data(service, portfolio_data):
    """Обновляет данные портфеля в Google Таблице с разделением по валюте и типу"""
    # Заголовки таблицы
    headers = [
        ["Тип", "Название", "Тикер", "ISIN", "Кол-во",
         "Ср. цена", "Тек. цена", "Валюта",
         "Годовой доход", "Доходность",
         "Прогноз. доход", "Тек. стоимость", "Прибыль"]
    ]

    # Расширенные заголовки с колонками для формул
    full_headers = headers[0] + ["Целевая цена", "Рекомендация"]

    # Категоризируем активы
    stocks_rub = [asset for asset in portfolio_data if asset["Тип актива"] == "Акция" and asset["Валюта"] == "rub"]
    stocks_hkd = [asset for asset in portfolio_data if asset["Тип актива"] == "Акция" and asset["Валюта"] == "hkd"]
    bonds_rub = [asset for asset in portfolio_data if asset["Тип актива"] == "Облигация" and asset["Валюта"] == "rub"]
    bonds_hkd = [asset for asset in portfolio_data if asset["Тип актива"] == "Облигация" and asset["Валюта"] == "hkd"]

    # Другие типы активов (ETF, валюта и т.д.)
    other_assets = [asset for asset in portfolio_data
                   if asset["Тип актива"] not in ["Акция", "Облигация"]
                   or asset["Валюта"] not in ["rub", "hkd"]]

    # Подготовка данных с разделением
    rows = []

    # Функция для добавления секции активов
    def add_asset_section(title, assets):
        if not assets:
            return

        rows.append([title, "", "", "", "", "", "", "", "", "", "", "", "", "", ""])
        rows.append(full_headers)

        for asset in assets:
            current_price = asset["Текущая цена"]
            dividend_yield = asset["Дивидендная/купонная доходность"]
            rows.append([
                asset["Тип актива"],
                asset["Название"],
                asset["Тикер"],
                asset["ISIN"],
                round(asset["Количество"], 4),
                round(asset["Средняя цена"], 4),
                round(current_price, 4),
                asset["Валюта"],
                round(asset["Годовой дивиденд/купон"], 4),
                f"{dividend_yield:.4%}" if dividend_yield > 0 else "0%",
                round(asset["Прогнозируемый годовой доход"], 2),
                round(asset["Текущая стоимость"], 2),
                round(asset["Прибыль/убыток"], 2),
                "", ""  # Пустые ячейки для формул
            ])

        # Итог для секции
        total_investment = sum(asset["Средняя цена"] * asset["Количество"] for asset in assets)
        total_current = sum(asset["Текущая стоимость"] for asset in assets)
        total_dividends = sum(asset["Прогнозируемый годовой доход"] for asset in assets)
        rows.append([
            f"ИТОГО {title}", "", "", "", "",
            "", "", "", "", "",
            round(total_dividends, 2),
            round(total_current, 2),
            round(total_current - total_investment, 2),
            "", ""
        ])
        rows.append(["", "", "", "", "", "", "", "", "", "", "", "", "", "", ""])  # Пустая строка

    # Добавляем секции в таблицу
    add_asset_section("АКЦИИ В РУБЛЯХ", stocks_rub)
    add_asset_section("АКЦИИ В HKD", stocks_hkd)
    add_asset_section("ОБЛИГАЦИИ В РУБЛЯХ", bonds_rub)
    add_asset_section("ОБЛИГАЦИИ В HKD", bonds_hkd)

    # Добавляем другие типы активов
    if other_assets:
        add_asset_section("ПРОЧИЕ АКТИВЫ", other_assets)

    # Общий итог
    total_investment = sum(asset["Средняя цена"] * asset["Количество"] for asset in portfolio_data)
    total_current = sum(asset["Текущая стоимость"] for asset in portfolio_data)
    total_dividends = sum(asset["Прогнозируемый годовой доход"] for asset in portfolio_data)
    rows.append([
        "ОБЩИЙ ИТОГ", "", "", "", "",
        "", "", "", "", "",
        round(total_dividends, 2),
        round(total_current, 2),
        round(total_current - total_investment, 2),
        "", ""
    ])

    # Формируем диапазон для записи
    range_name = f"{SHEET_NAME}!A1"
    # Очищаем старые данные
    clear_sheet(service, f"{SHEET_NAME}!A1:Z1000")

    # Записываем данные без формул
    body = {
        'values': rows
    }

    try:
        result = service.spreadsheets().values().update(
            spreadsheetId=SPREADSHEET_ID,
            range=range_name,
            valueInputOption='USER_ENTERED',
            body=body
        ).execute()
        print(f"✅ Успешно обновлено {result.get('updatedCells')} ячеек")

        # Теперь добавляем формулы для каждой секции отдельно
        current_row = 1
        all_sections = [
            ("АКЦИИ В РУБЛЯХ", stocks_rub),
            ("АКЦИИ В HKD", stocks_hkd),
            ("ОБЛИГАЦИИ В РУБЛЯХ", bonds_rub),
            ("ОБЛИГАЦИИ В HKD", bonds_hkd),
            ("ПРОЧИЕ АКТИВЫ", other_assets)
        ]

        for section_title, section_assets in all_sections:
            if not section_assets:
                continue

            # Ищем начало секции в таблице
            while current_row <= len(rows):
                if rows[current_row-1][0] == section_title:
                    break
                current_row += 1

            # Пропускаем заголовок секции и заголовки колонок
            current_row += 2

            # Добавляем формулы для активов в секции
            if len(section_assets) > 0:
                formulas = []
                for i in range(len(section_assets)):
                    formulas.append([
                        f"=F{current_row+i}*0.9",
                        f"=ЕСЛИ(G{current_row+i}<N{current_row+i};\"Купить\";ЕСЛИ(G{current_row+i}>F{current_row+i}*1.1;\"Продать\";\"Держать\"))"
                    ])

                formula_body = {
                    'values': formulas
                }

                formula_range = f"{SHEET_NAME}!N{current_row}:O{current_row + len(section_assets) - 1}"
                service.spreadsheets().values().update(
                    spreadsheetId=SPREADSHEET_ID,
                    range=formula_range,
                    valueInputOption='USER_ENTERED',
                    body=formula_body
                ).execute()

                # Переходим к следующей секции (пропускаем итоговую строку и пустую строку)
                current_row += len(section_assets) + 2

        print("✅ Формулы для анализа добавлены корректно для всех секций")
        return True
    except HttpError as error:
        # Обработка ошибки превышения квоты
        if error.resp.status == 403 and "storage quota" in str(error).lower():
            print("❌ ОШИБКА: Превышена квота Google Диска!")
            print("РЕШЕНИЕ: 1. Удалите старые таблицы 2. Используйте существующую таблицу")
        else:
            print(f"❌ Ошибка Google Sheets API: {error}")
        return False

In [24]:
def get_google_sheets_service():
    """Создает сервис Google Sheets с использованием сервисного аккаунта"""
    credentials = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE, scopes=SCOPES)
    service = build('sheets', 'v4', credentials=credentials)
    return service

def main():
    """Основная функция"""
    print("🚀 Начало обновления инвестиционного портфеля")

    try:
        # Создаем сервис Google Sheets
        service = get_google_sheets_service()
        print("✅ Подключение к Google Sheets установлено")

        # Получаем данные портфеля (замените на вызов Tinkoff API)
        portfolio_data = getPortfolioInfo()
        print(f"📊 Получено данных об активах: {len(portfolio_data)}")

        # Обновляем таблицу
        success = update_portfolio_data(service, portfolio_data)

        if success:
            print("🎉 Обновление завершено успешно!")
        else:
            print("⚠️ Обновление завершено с ошибками")

    except Exception as e:
        print(f"❌ Критическая ошибка: {str(e)}")
        print("Проверьте:")
        print("1. Наличие файла credentials.json")
        print("2. Правильность SPREADSHEET_ID")
        print("3. Доступ к таблице для сервисного аккаунта")

if __name__ == "__main__":
    main()

🚀 Начало обновления инвестиционного портфеля
✅ Подключение к Google Sheets установлено
📊 Получено данных об активах: 70
✅ Диапазон Portfolio!A1:Z1000 очищен
✅ Успешно обновлено 1305 ячеек
✅ Формулы для анализа добавлены корректно для всех секций
🎉 Обновление завершено успешно!
